# Lesson 06 Lab — Profiling Mixed Precision and Verifying Dispatch

**Puzzle:** If autocast made an operation faster, does that prove the intended low-precision kernel ran?

The saved outputs were generated by executing every code cell on the recorded RTX 5090. Run all cells to regenerate the evidence on your own CUDA GPU.

## 0. Predict before running

Write down: (1) the expected direction, (2) the mechanism, (3) the observation that would reverse your prediction, and (4) the evidence level required for the claim.

## 1. Theory — objects and data flow

Three evidence layers answer different questions: model outputs show semantic effect, framework operators show graph dispatch, and native kernel traces show the implementation actually launched.

### Core mechanism

Profiling can expose casts, copies, GEMMs, launch count, and device time. Warm-up is required because lazy initialization, compilation, and allocator growth are not steady-state execution.

In [1]:
from pathlib import Path
import json
import sys
import torch

chapter_rel = Path("chapters/01-mixed-precision-int4")
repo_root = next(
    p for p in [Path.cwd(), *Path.cwd().parents]
    if (p / chapter_rel / "support" / "lab_common.py").exists()
)
sys.path.insert(0, str(repo_root / chapter_rel / "support"))
from lab_common import (base_result, cuda_benchmark, environment_record,
                        error_metrics, require_cuda, save_result,
                        symmetric_quantize)

lesson_dir = repo_root / chapter_rel / "06-mixed-precision-profiling"
device = require_cuda()
torch.manual_seed(2026 + 6)
environment = environment_record()
print(json.dumps(environment, indent=2))


{
  "gpu": "NVIDIA GeForce RTX 5090",
  "compute_capability": "12.0",
  "gpu_memory_gib": 31.358,
  "python": "3.12.13",
  "torch": "2.12.0",
  "cuda_runtime": "13.0"
}


## 2. Connect theory to the experiment

### Engineering trade-off

A detailed profiler perturbs runtime and creates large traces; a light CUDA-event benchmark has lower overhead but less attribution. Use the least intrusive tool that can answer the current claim.

### What this code tests

The notebook pairs repeated CUDA-event timing with selected PyTorch profiler events and explicitly stops short of inventing a native kernel name.

**Experiment:** Profile an autocast BF16 GEMM with PyTorch Profiler and record the relevant operator events beside CUDA-event timing.

**Declared evidence label:** `pytorch-gpu`. Check that the shapes, controlled variables, and units match the theoretical question before executing.

In [2]:
from torch.profiler import ProfilerActivity, profile
import warnings
a = torch.randn(2048, 2048, device=device); b = torch.randn(2048, 2048, device=device)
def work():
    with torch.autocast("cuda", dtype=torch.bfloat16): return a @ b
timing = cuda_benchmark(work, warmup=5, repeats=15)
with warnings.catch_warnings():
    warnings.filterwarnings("ignore", message=".*Profiler clears events.*")
    with profile(activities=[ProfilerActivity.CPU, ProfilerActivity.CUDA]) as prof:
        for _ in range(3): work()
torch.cuda.synchronize()
events = []
for e in prof.key_averages():
    if any(k in e.key.lower() for k in ("mm", "matmul", "to", "copy")):
        events.append({"operator": e.key, "count": e.count})
result = base_result(6, "pytorch-gpu"); result.update({"shape": [2048, 2048], "timing": timing,
    "pytorch_operator_events": events[:20], "conclusion": "Autocast timing and PyTorch operator evidence were captured; native kernel identity was not claimed."})


## 3. Inspect the evidence

Require both repeated timing and trace evidence. This lab deliberately labels PyTorch operators rather than claiming a native kernel name.

### Acceptance and rollback gate

First reproduce timing without a profiler, then capture a short aligned trace. Name only the level actually observed: PyTorch operator, CUDA kernel, or end-to-end phase.

In [3]:
artifact_path = save_result(result, lesson_dir)
print(json.dumps(result, indent=2, sort_keys=True))
print("Saved: artifacts/rtx5090-result.json")


{
  "conclusion": "Autocast timing and PyTorch operator evidence were captured; native kernel identity was not claimed.",
  "environment": {
    "compute_capability": "12.0",
    "cuda_runtime": "13.0",
    "gpu": "NVIDIA GeForce RTX 5090",
    "gpu_memory_gib": 31.358,
    "python": "3.12.13",
    "torch": "2.12.0"
  },
  "evidence_label": "pytorch-gpu",
  "executed_at_utc": "2026-08-07T14:49:58+00:00",
  "lesson": 6,
  "pytorch_operator_events": [
    {
      "count": 6,
      "operator": "aten::matmul"
    },
    {
      "count": 6,
      "operator": "aten::to"
    },
    {
      "count": 6,
      "operator": "aten::_to_copy"
    },
    {
      "count": 6,
      "operator": "aten::copy_"
    },
    {
      "count": 3,
      "operator": "aten::mm"
    }
  ],
  "schema_version": 1,
  "shape": [
    2048,
    2048
  ],
  "timing": {
    "median_ms": 0.104416,
    "p90_ms": 0.10624,
    "repeats": 15,
    "samples_ms": [
      0.115168,
      0.10736,
      0.105472,
      0.10624,
    

## 4. Explain the result

Use a two-part proof: controlled timing for effect and profiler evidence for dispatch; escalate to Nsight for native-kernel claims.

Relate the measured fields back to the mechanism above. Treat the checked-in result as one hardware/software observation, not a universal ranking. The complete derivation, evidence boundary, and primary references are in [`README.md`](README.md).